# 📈 Notebook 2 — Tuning W and R: Availability vs Consistency

Strong consistency (`W + R > N`) is nice, but it has a cost: you need more
replicas to respond to every operation. If too many nodes are down, the
operation **fails**.

In this notebook we **quantify** that trade-off with a tiny probability model
and a couple of plots.


## 🎲 The model

Assume each of the `N` replicas is **independently up** with probability `p`
(say, `p = 0.95`). A write with threshold `W` succeeds iff **at least `W`**
replicas are up.

The probability of that is the tail of a binomial:

$$
P(\text{write ok}) = \sum_{k=W}^{N} \binom{N}{k}\, p^{k}\,(1-p)^{N-k}
$$

Same formula works for reads with threshold `R`.


In [ ]:
from math import comb

def prob_success(n, quorum, p_up):
    '''P(at least `quorum` out of `n` replicas are up).'''
    return sum(
        comb(n, k) * (p_up ** k) * ((1 - p_up) ** (n - k))
        for k in range(quorum, n + 1)
    )

# Sanity check: quorum=1 with any p>0 is nearly always available
print(f"N=5, W=1, p=0.9 → {prob_success(5, 1, 0.9):.4f}")
print(f"N=5, W=5, p=0.9 → {prob_success(5, 5, 0.9):.4f}   (need ALL up)")
print(f"N=5, W=3, p=0.9 → {prob_success(5, 3, 0.9):.4f}")

# Closed forms we can check exactly, so a typo in the binomial shows up immediately.
assert abs(prob_success(5, 5, 0.9) - 0.9 ** 5) < 1e-12          # all five must be up
assert abs(prob_success(5, 1, 0.9) - (1 - 0.1 ** 5)) < 1e-12    # at least one is up
assert abs(prob_success(5, 0, 0.9) - 1.0) < 1e-12               # a quorum of 0 always succeeds
# Availability must fall monotonically as the quorum grows.
avail = [prob_success(5, q, 0.9) for q in range(1, 6)]
assert avail == sorted(avail, reverse=True), avail


## 🧪 Availability vs quorum size (N = 5)

In [ ]:
import matplotlib.pyplot as plt

N = 5
p_values = [0.99, 0.95, 0.9, 0.8]
quorums = list(range(1, N + 1))

fig, ax = plt.subplots(figsize=(7, 4))
for p in p_values:
    ax.plot(quorums, [prob_success(N, q, p) for q in quorums],
            marker="o", label=f"p_up = {p}")
ax.set_xlabel("Quorum size (W or R)")
ax.set_ylabel("P(operation succeeds)")
ax.set_title(f"Availability vs quorum size  (N = {N})")
ax.set_xticks(quorums)
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


**Reading the chart**

- `W = 1` is essentially always available — but pairs with `R = N` if you want
  strong consistency, which tanks read availability.
- `W = N` needs **every** replica up; one flaky node and writes fail.
- The sweet spot for many systems is `W = R = ⌈(N+1)/2⌉` (the **majority
  quorum**). For `N=5` that's 3/3.


## 🌐 Real-world picks

| System        | Default N | Default W | Default R | Notes |
|---------------|-----------|-----------|-----------|-------|
| DynamoDB      | 3 (per partition) | 2 | 1 (eventual) / 2 (strong) | user chooses per-request |
| Cassandra     | configurable RF | `LOCAL_QUORUM`, `ONE`, `ALL`, … | same | per-query tunable |
| Riak          | 3 | 2 | 2 | classic Dynamo defaults |
| etcd / ZK     | 3 or 5 | majority | majority | **Paxos/Raft**, not Dynamo-quorum |

Note: **Raft/Paxos systems (etcd, ZooKeeper, Consul) also use majority
quorums**, but they add a leader and log replication on top — different
mechanism, same "quorum overlap" idea at the core.


## 💡 Latency trade-off (quick simulation)

Write latency ≈ time until the **W-th** replica acks. If each replica's ack
time is random (say exponential), bigger `W` means you wait longer because
you're waiting on the slowest of many.


In [ ]:
import random, statistics

def simulate_write_latency(N, W, mean_ms=10, trials=5000, seed=0):
    rnd = random.Random(seed)
    latencies = []
    for _ in range(trials):
        acks = sorted(rnd.expovariate(1 / mean_ms) for _ in range(N))
        latencies.append(acks[W - 1])         # W-th fastest = we can return
    return statistics.median(latencies), statistics.quantiles(latencies, n=20)[-1]

print(f"{'W':>2} | {'median (ms)':>11} | {'p95 (ms)':>8}")
print("-" * 30)
rows = {}
for W in range(1, 6):
    med, p95 = simulate_write_latency(N=5, W=W)
    rows[W] = (med, p95)
    print(f"{W:>2} | {med:>11.2f} | {p95:>8.2f}")

# Waiting for the W-th of N acks is an order statistic: it can only grow with W.
# Both the median and the tail must be monotone, and the tail must grow faster.
meds = [rows[W][0] for W in range(1, 6)]
p95s = [rows[W][1] for W in range(1, 6)]
assert meds == sorted(meds), meds
assert p95s == sorted(p95s), p95s
assert (p95s[-1] - p95s[0]) > (meds[-1] - meds[0]), "the tail should stretch more than the median"
print(f"\n✔ going from W=1 to W=5 costs {meds[-1]-meds[0]:.1f} ms at the median "
      f"but {p95s[-1]-p95s[0]:.1f} ms at p95 — the tail is where quorums hurt")


Bigger `W` → higher tail latency. This is the **tail-at-scale** problem.
Cassandra's `speculative_retry` and DynamoDB's hedged requests fight it.

## 🎯 Exercise

1. Plot the same availability curve for `N = 3` and `N = 7`. When does
   increasing `N` **hurt** write availability? (Hint: only if you also raise
   `W`.)
2. For `N = 5`, `p = 0.95`, what's the probability **both** a `W=3` write
   **and** an `R=3` read succeed in the same request? (Assume independence.)
